# Coexistence phase diagram

ProP Gen theory predicts regimes in which several genotype-phenotype pairs
coexist, with the *ordering* of their equilibrium frequencies changing as the
mutation rate and the genotype-phenotype map vary. Each distinct ordering is a
phase.

This is computed **exactly and analytically** from `propgen.phase_diagram`,
which evaluates the Perron-Frobenius equilibrium at every grid point. No
simulation is involved and no stored data is needed: the grid below takes
under a second.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path

from propgen import Landscape, load_config, phase_diagram, resolve_config
from propgen.plotting import set_paper_style

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

set_paper_style()
FIGDIR = REPO / "figures"
FIGDIR.mkdir(exist_ok=True)

We use the buoying landscape: two genotypes, two phenotypes, with genotype 1's
phenotype distribution swept as `[phi, 1 - phi]`.

In [ ]:
landscape = resolve_config(load_config(REPO / "configs" / "buoy.yaml"))["landscape"]
landscape

In [ ]:
N_GRID = 100

diagram = phase_diagram(
    landscape,
    mutation_rates=np.linspace(0.005, 0.5, N_GRID),
    pheno_prob_values=np.linspace(0.0, 1.0, N_GRID),
    genotype=1,
)
print(diagram)
for i in range(len(diagram.orderings)):
    print(f"  phase {i}: {diagram.label(i, landscape.n_phenotypes)}")

## The diagram

In [ ]:
n_phases = len(diagram.orderings)
cmap = ListedColormap(plt.cm.tab20(np.linspace(0, 1, 20))[:n_phases])

fig, ax = plt.subplots(figsize=(9, 6))
ax.pcolormesh(
    diagram.pheno_prob_values,
    diagram.mutation_rates,
    diagram.phase,
    cmap=cmap,
    vmin=-0.5,
    vmax=n_phases - 0.5,
    shading="auto",
)

ax.set_xlabel(r"$\phi_1^{(0)}$")
ax.set_ylabel(r"Mutation rate $\mu$")

handles = [
    Patch(facecolor=cmap(i), label=diagram.label(i, landscape.n_phenotypes))
    for i in range(n_phases)
]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=11)

plt.tight_layout()
plt.savefig(FIGDIR / "phasediag.pdf")
plt.show()

## Frequencies underlying the phases

The phase boundaries are where two of these curves cross.

In [ ]:
MU = 0.1
row = int(np.argmin(np.abs(diagram.mutation_rates - MU)))
Ng, Np = landscape.shape

plt.figure(figsize=(8, 5))
for g in range(Ng):
    for p in range(Np):
        plt.plot(diagram.pheno_prob_values, diagram.f_eq[row, :, g * Np + p],
                 lw=2, label=f"$f_{{{g}}}^{{({p})}}$")

plt.xlabel(r"$\phi_1^{(0)}$")
plt.ylabel("Equilibrium frequency")
plt.title(rf"$\mu$ = {diagram.mutation_rates[row]:.3f}")
plt.legend()
plt.tight_layout()
plt.savefig(FIGDIR / "phasediag_frequencies.pdf")
plt.show()